# Week 13: Amazon Bedrock — Managed Models, Knowledge Bases & Synthetic Data

## Learning Objectives

By the end of this session, you will be able to:
1. **Call Amazon Bedrock models** using the unified Converse API (Claude, Llama, Nova)
2. **Query a Knowledge Base** for grounded, document-backed answers
3. **Generate synthetic training data** using LLMs for downstream ML tasks
4. **Use LLMs for exploratory data analysis** and evaluate outputs with DeepEval

## Prerequisites

- Completed Week 11 (OpenAI API, prompting patterns) and Week 12 (local models, evaluation)
- Watched pre-class videos on Amazon Bedrock, available models, Knowledge Bases concepts
- Running this notebook in **AWS SageMaker** (your execution role already has Bedrock permissions)

## Session Format (~2 hours)

| Section | Duration | Type |
|---------|----------|------|
| Section 0: Setup & AWS Auth | 5 min | Code |
| Section 1: Bedrock Converse API | 25 min | Demo-heavy |
| Lab 1: Bedrock Model Showdown | 15 min | Lab |
| Section 2: Knowledge Bases | 20 min | Demo |
| Lab 2: Query the Knowledge Base | 15 min | Lab |
| Section 3: Synthetic Data Generation | 15 min | Demo |
| Lab 3: Generate Fraud Training Data | 15 min | Lab |
| Section 4: LLM-Assisted EDA + DeepEval | 15 min | Demo |
| Wrap-up & Homework | 5 min | Markdown |

## What We'll Build Today

Continuing our fraud detection story, we take the same classification task to
**Amazon Bedrock** — and this time we go beyond just classifying. By the end of
today, you'll have a complete pipeline: from model comparison, to policy-grounded
answers, to generating the synthetic training data that Week 14 will use for
fine-tuning.

![Week 13 Overview Flow](charts/overview_flow.png)

**One thread, one goal**: Every section today feeds into the next, culminating
in a validated synthetic dataset ready for Week 14's fine-tuning.

## Environment

**Platform**: AWS SageMaker  
GPU is **not required** — all work is API-based. boto3 picks up your SageMaker
execution role automatically (no API keys needed).

# Section 0: Environment Setup

We'll use boto3 to call Amazon Bedrock APIs and DeepEval for evaluation.

In [ ]:
# =============================================================================
# INSTALL REQUIRED LIBRARIES
# =============================================================================
# boto3: AWS SDK for Python (already installed on SageMaker)
# deepeval: LLM evaluation framework with Bedrock integration
# aiobotocore: Required by DeepEval's AmazonBedrockModel

!pip install -q deepeval aiobotocore

# =============================================================================
# IMPORTS
# =============================================================================
import boto3                               # AWS SDK
import json                                # JSON parsing
import os                                  # Environment variables
import time                                # Latency measurement
import re                                  # Regex for parsing
import numpy as np                         # Numerical operations
import pandas as pd                        # Data manipulation
import matplotlib.pyplot as plt            # Visualization
from collections import Counter            # Counting results

# =============================================================================
# VERIFY INSTALLATIONS
# =============================================================================
print("Library versions:")
print(f"  boto3:   {boto3.__version__}")
print("\n✅ All libraries installed successfully!")

In [ ]:
# =============================================================================
# AWS CREDENTIALS SETUP (SageMaker)
# =============================================================================
# You're running this notebook in SageMaker Studio/Notebooks, which means
# boto3 automatically picks up your execution role — no credentials needed!
# Your instructor has already attached Bedrock permissions to your role.

AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

# Create Bedrock Runtime client (for model inference)
bedrock_runtime = boto3.client('bedrock-runtime', region_name=AWS_REGION)

# Create Bedrock Agent Runtime client (for Knowledge Bases)
bedrock_agent_runtime = boto3.client('bedrock-agent-runtime', region_name=AWS_REGION)

# Verify connection
try:
    bedrock_client = boto3.client('bedrock', region_name=AWS_REGION)
    models = bedrock_client.list_foundation_models()
    model_count = len(models['modelSummaries'])
    print(f"Region: {AWS_REGION}")
    print(f"\n✅ Connected to Bedrock! {model_count} models available.")
except Exception as e:
    print(f"\n❌ Connection failed — check with your instructor: {e}")
    print("   Make sure you're running this from SageMaker with Bedrock permissions.")

In [ ]:
# =============================================================================
# FRAUD TRANSACTION DATASET — EXPANDED (50 transactions)
# =============================================================================
# We expand from the 8 transactions used in Weeks 11-12 to 50 for better
# evaluation. The original 8 are still included (TXN-001 through TXN-008).

transaction_descriptions = [
    # --- Original 8 from Weeks 11-12 ---
    {"id": "TXN-001", "description": "Customer reports unauthorized wire transfer of $4,500 to unknown overseas account. No prior international transaction history. Transfer initiated at 3:47 AM local time.", "actual_label": "fraud"},
    {"id": "TXN-002", "description": "Regular monthly payment of $89.99 to Netflix streaming service. Consistent with 18-month subscription history. Payment from primary checking account.", "actual_label": "legitimate"},
    {"id": "TXN-003", "description": "Three consecutive ATM withdrawals totaling $1,500 in different cities within 2 hours. Card was reported lost the following day. Withdrawals at non-bank ATMs.", "actual_label": "fraud"},
    {"id": "TXN-004", "description": "Online purchase of $234.56 at Amazon.com for household electronics. Shipping to address on file. Customer has frequent Amazon purchase history.", "actual_label": "legitimate"},
    {"id": "TXN-005", "description": "Customer disputes charge of $2,100 at luxury jewelry store in Miami. Customer's location confirmed as Chicago at time of purchase. No travel alerts set.", "actual_label": "fraud"},
    {"id": "TXN-006", "description": "Automatic payroll direct deposit of $3,245.67 from employer ABC Corp. Matches bi-weekly pay schedule. Amount consistent with employment records.", "actual_label": "legitimate"},
    {"id": "TXN-007", "description": "Multiple small online purchases ($5-$15) at various digital stores within 30 minutes. None of these merchants appear in customer's history. Different IP addresses used.", "actual_label": "fraud"},
    {"id": "TXN-008", "description": "Grocery purchase of $67.23 at Whole Foods Market. Customer shops here weekly based on 2-year transaction history. Paid with debit card at POS terminal.", "actual_label": "legitimate"},

    # --- Additional fraud transactions (TXN-009 through TXN-025) ---
    {"id": "TXN-009", "description": "Account password changed and $8,200 transferred to a new payee within 15 minutes. Login originated from an IP address in a different country than the account holder's residence.", "actual_label": "fraud"},
    {"id": "TXN-010", "description": "Credit card used for $3,400 purchase at electronics store in Lagos, Nigeria. Cardholder has never traveled outside the United States. Card was not reported stolen.", "actual_label": "fraud"},
    {"id": "TXN-011", "description": "Five gift card purchases of $500 each at different Walmart locations within one hour. Customer has no prior gift card purchase history. All purchases made with the same debit card.", "actual_label": "fraud"},
    {"id": "TXN-012", "description": "Online gambling deposit of $2,000 to an unlicensed offshore betting site. Customer's account shows no prior gambling-related transactions. Deposit made at 4:12 AM.", "actual_label": "fraud"},
    {"id": "TXN-013", "description": "Wire transfer of $15,000 to a recently created account at a foreign bank. Transfer requested via phone call, but customer's voice did not match voiceprint on file.", "actual_label": "fraud"},
    {"id": "TXN-014", "description": "Rapid succession of 12 declined transactions followed by one approved transaction of $1,899 at an electronics retailer. Different card numbers attempted from same IP.", "actual_label": "fraud"},
    {"id": "TXN-015", "description": "Customer's debit card used for contactless payment of $750 at a gas station in Texas while customer was checked into a hotel in New York. No travel notification on file.", "actual_label": "fraud"},
    {"id": "TXN-016", "description": "Account drained of $6,300 through a series of Zelle transfers to three unknown recipients within 20 minutes. Customer claims they did not authorize the transfers.", "actual_label": "fraud"},
    {"id": "TXN-017", "description": "Purchase of $4,200 in cryptocurrency from an unregulated exchange using a newly added payment method. Account security questions were changed 30 minutes prior.", "actual_label": "fraud"},
    {"id": "TXN-018", "description": "Two simultaneous transactions: $1,100 at a restaurant in London and $890 at a store in Sydney. Physical card required at both locations. Cardholder resides in Boston.", "actual_label": "fraud"},
    {"id": "TXN-019", "description": "Refund of $2,500 processed to a different card than the original purchase. Original purchase was made 8 months ago. Refund requested through customer service chat.", "actual_label": "fraud"},
    {"id": "TXN-020", "description": "Cash advance of $3,000 at an ATM in a high-risk neighborhood at 2:30 AM. Customer's account has never had a cash advance. PIN was entered correctly on first attempt.", "actual_label": "fraud"},
    {"id": "TXN-021", "description": "Online purchase of $5,600 worth of designer handbags from a suspicious website with no SSL certificate. Shipping address differs from billing address and is a P.O. box.", "actual_label": "fraud"},
    {"id": "TXN-022", "description": "Authorized user added to account and immediately made a $7,500 purchase. The authorized user's identity could not be verified through standard KYC checks.", "actual_label": "fraud"},
    {"id": "TXN-023", "description": "Series of micro-transactions ($0.01 to $1.00) at 15 different online merchants within 5 minutes. Pattern consistent with card testing before a larger fraudulent purchase.", "actual_label": "fraud"},
    {"id": "TXN-024", "description": "Balance transfer of $12,000 to a new credit card opened the same day. Application used a slightly different spelling of the customer's name and a different phone number.", "actual_label": "fraud"},
    {"id": "TXN-025", "description": "Purchase of $950 airline ticket to a one-way international destination booked 2 hours before departure. Customer has no passport on file and no prior international travel.", "actual_label": "fraud"},

    # --- Additional legitimate transactions (TXN-026 through TXN-050) ---
    {"id": "TXN-026", "description": "Monthly mortgage payment of $1,847.33 to Wells Fargo Home Mortgage. Amount unchanged for 3 years. Auto-debit from primary checking account on the 1st of each month.", "actual_label": "legitimate"},
    {"id": "TXN-027", "description": "Quarterly insurance premium of $412.00 to State Farm. Consistent with 5-year policy history. Payment matches scheduled auto-pay date.", "actual_label": "legitimate"},
    {"id": "TXN-028", "description": "Gas station purchase of $52.18 at Shell on Highway 101. Customer fills up at this location every Friday afternoon. Debit card used at pump with PIN.", "actual_label": "legitimate"},
    {"id": "TXN-029", "description": "Online subscription renewal of $14.99 for Spotify Premium. Same charge every month for the past 2 years. Billed to Visa ending in 4532.", "actual_label": "legitimate"},
    {"id": "TXN-030", "description": "Restaurant charge of $78.45 at Olive Garden in customer's hometown. Tip of 20% added. Customer dines here approximately twice per month.", "actual_label": "legitimate"},
    {"id": "TXN-031", "description": "Utility bill payment of $156.78 to ConEdison via online banking. Amount within normal seasonal range. Payment made 3 days before due date as usual.", "actual_label": "legitimate"},
    {"id": "TXN-032", "description": "Daycare tuition payment of $1,200.00 to Little Stars Learning Center. Same amount every two weeks since January. Matches enrollment records.", "actual_label": "legitimate"},
    {"id": "TXN-033", "description": "Pharmacy purchase of $23.45 at CVS near customer's home address. Customer has weekly prescription pickups at this location. Paid with FSA debit card.", "actual_label": "legitimate"},
    {"id": "TXN-034", "description": "Annual gym membership renewal of $599.00 at Planet Fitness. Same charge last year at the same time. Customer checks in 4-5 times per week.", "actual_label": "legitimate"},
    {"id": "TXN-035", "description": "Transfer of $500.00 to savings account at same bank. Customer makes this transfer on the 15th of every month. Part of automatic savings plan.", "actual_label": "legitimate"},
    {"id": "TXN-036", "description": "Online order of $145.67 at Target.com. Shipping to home address on file. Customer has Target Circle membership and shops online monthly.", "actual_label": "legitimate"},
    {"id": "TXN-037", "description": "Car payment of $423.56 to Toyota Financial Services. Amount matches lease agreement. Auto-pay set up 18 months ago, never missed a payment.", "actual_label": "legitimate"},
    {"id": "TXN-038", "description": "Coffee shop purchase of $6.75 at Starbucks near customer's office. Customer visits this location every weekday morning. Mobile order via app.", "actual_label": "legitimate"},
    {"id": "TXN-039", "description": "Charitable donation of $100.00 to American Red Cross. Customer makes this donation annually in December. Tax-deductible receipt issued.", "actual_label": "legitimate"},
    {"id": "TXN-040", "description": "Veterinary bill of $287.50 at Banfield Pet Hospital. Customer has a pet wellness plan. Visit scheduled 2 weeks ago for annual checkup.", "actual_label": "legitimate"},
    {"id": "TXN-041", "description": "Home improvement purchase of $342.89 at Home Depot. Customer recently purchased a home and has made 6 Home Depot purchases in the past month. In-store chip payment.", "actual_label": "legitimate"},
    {"id": "TXN-042", "description": "Monthly student loan payment of $567.89 to Navient. Same amount for 4 years. Auto-debit on the 5th of each month from checking account.", "actual_label": "legitimate"},
    {"id": "TXN-043", "description": "Dry cleaning pickup charge of $34.50 at Express Cleaners. Customer drops off clothes every Monday and picks up Wednesday. Store is 2 blocks from home.", "actual_label": "legitimate"},
    {"id": "TXN-044", "description": "Online purchase of $89.99 for annual antivirus software renewal from Norton. Same charge last year. License key sent to email on file.", "actual_label": "legitimate"},
    {"id": "TXN-045", "description": "Ride-share charge of $24.67 from Uber. Trip from customer's office to home address. Customer uses Uber 2-3 times per week for commute.", "actual_label": "legitimate"},
    {"id": "TXN-046", "description": "Wire transfer of $2,000 to customer's own account at another bank. Both accounts have been linked for 3 years. Transfer initiated via verified mobile app.", "actual_label": "legitimate"},
    {"id": "TXN-047", "description": "Hotel charge of $189.00 at Marriott in San Francisco. Customer has a confirmed reservation matching the dates. Corporate travel card used, expense report filed.", "actual_label": "legitimate"},
    {"id": "TXN-048", "description": "Lawn care service payment of $75.00 to GreenScape LLC. Same vendor, same amount, every two weeks from April through October for 3 years.", "actual_label": "legitimate"},
    {"id": "TXN-049", "description": "Birthday gift purchase of $65.00 at Barnes & Noble online. Shipping to a different address (gift recipient). Customer makes similar purchases around family birthdays.", "actual_label": "legitimate"},
    {"id": "TXN-050", "description": "Tollway auto-replenishment charge of $40.00 to I-PASS. Triggered when balance fell below $10 threshold. Customer commutes daily on the tollway.", "actual_label": "legitimate"},
]

df = pd.DataFrame(transaction_descriptions)
fraud_count = (df['actual_label'] == 'fraud').sum()
legit_count = (df['actual_label'] == 'legitimate').sum()
print(f"Loaded {len(df)} transactions ({fraud_count} fraud, {legit_count} legitimate)")
print(f"\nOriginal 8 from Weeks 11-12: TXN-001 through TXN-008")
print(f"New fraud examples: TXN-009 through TXN-025 (17 new)")
print(f"New legitimate examples: TXN-026 through TXN-050 (25 new)")

# Section 1: Amazon Bedrock — The Converse API

In Week 11 you called OpenAI and Anthropic APIs directly. In Week 12 you ran
models locally with HuggingFace. Now we'll use **Amazon Bedrock** — AWS's
managed service that gives you access to multiple model providers through a
**single unified API**.

## Why Bedrock?

| Approach | Provider | Pros | Cons |
|----------|----------|------|------|
| **OpenAI API** (Week 11) | OpenAI | Best quality, easy API | Data leaves your org, vendor lock-in |
| **Local HuggingFace** (Week 12) | You | Free, private, fast | Limited quality, need GPU |
| **Amazon Bedrock** (Today) | AWS | Enterprise compliance, multiple providers, single API | AWS ecosystem, cost |

## Available Models on Bedrock

| Model | Provider | Model ID | Strengths |
|-------|----------|----------|-----------|
| Claude 3 Haiku | Anthropic | `anthropic.claude-sonnet-4-5-20250929-v1:0` | Fast, cheap, good quality |
| Claude 3.5 Sonnet | Anthropic | `anthropic.claude-3-5-sonnet-20241022-v2:0` | Best quality on Bedrock |
| Llama 3.1 70B | Meta | `us.meta.llama3-1-70b-instruct-v1:0` | Open-source, good reasoning |
| Nova Lite | Amazon | `amazon.nova-lite-v1:0` | Cheapest, fastest |
| Nova Pro | Amazon | `amazon.nova-pro-v1:0` | Good balance of cost/quality |
| Titan Text Express | Amazon | `amazon.titan-text-express-v1` | AWS-native, low cost |

> **Note**: Llama models on Bedrock require an *inference profile* prefix (`us.`)
> because Meta models are accessed via cross-region inference profiles, not direct
> model endpoints. You'll see `us.meta.llama3-...` instead of `meta.llama3-...`.

### The Converse API

Bedrock's `converse` API is **model-agnostic** — the same code works with
any model. Just change the `modelId`. This is huge: no more learning different
APIs for each provider.

In [ ]:
# =============================================================================
# DEMO: First Bedrock Call — Claude 3 Haiku
# =============================================================================
# The converse API is the recommended way to call Bedrock models.
# It provides a unified interface across all model providers.

sample = transaction_descriptions[0]  # TXN-001: Unauthorized wire transfer

# Build the prompt (same classification task as Weeks 11-12)
prompt = (
    "Classify the following bank transaction as 'fraud' or 'legitimate'. "
    "Respond with only one word: fraud or legitimate.\n\n"
    f'Transaction: "{sample["description"]}"\n\n'
    "Classification:"
)

# Call Bedrock with the Converse API
start = time.time()
response = bedrock_runtime.converse(
    modelId='anthropic.claude-sonnet-4-5-20250929-v1:0',
    messages=[
        {
            'role': 'user',
            'content': [{'text': prompt}]
        }
    ],
    inferenceConfig={
        'maxTokens': 10,
        'temperature': 0.0,
    }
)
elapsed = time.time() - start

# Extract the response
output_text = response['output']['message']['content'][0]['text'].strip().lower()
input_tokens = response['usage']['inputTokens']
output_tokens = response['usage']['outputTokens']
stop_reason = response['stopReason']

print(f"Transaction: {sample['id']}")
print(f"Prediction:  '{output_text}'")
print(f"Actual:      '{sample['actual_label']}'")
print(f"Latency:     {elapsed:.2f}s")
print(f"Tokens:      {input_tokens} in / {output_tokens} out")
print(f"Stop reason: {stop_reason}")
print(f"\n💡 The Converse API returns structured metadata: token counts,")
print(f"   stop reason, and more. This is great for cost tracking!")

In [ ]:
# =============================================================================
# DEMO: Model-Agnostic API — Same Code, Different Models
# =============================================================================
# The beauty of Converse: just change the modelId.

MODELS = {
    'Claude Haiku': 'anthropic.claude-sonnet-4-5-20250929-v1:0',
    'Nova Lite': 'amazon.nova-lite-v1:0',
    'Llama 3.1 70B': 'us.meta.llama3-1-70b-instruct-v1:0',  # inference profile prefix required
}

sample = transaction_descriptions[0]
prompt = (
    "Classify the following bank transaction as 'fraud' or 'legitimate'. "
    "Respond with only one word: fraud or legitimate.\n\n"
    f'Transaction: "{sample["description"]}"\n\n'
    "Classification:"
)

for model_name, model_id in MODELS.items():
    start = time.time()
    try:
        response = bedrock_runtime.converse(
            modelId=model_id,
            messages=[{'role': 'user', 'content': [{'text': prompt}]}],
            inferenceConfig={'maxTokens': 10, 'temperature': 0.0}
        )
        elapsed = time.time() - start
        output = response['output']['message']['content'][0]['text'].strip().lower()
        tokens_in = response['usage']['inputTokens']
        tokens_out = response['usage']['outputTokens']
        print(f"{model_name:20s}: '{output}' | {elapsed:.2f}s | {tokens_in}+{tokens_out} tokens")
    except Exception as e:
        print(f"{model_name:20s}: ❌ Error — {e}")

print(f"\n💡 Same code, same prompt, three different models.")
print(f"   In production, you could A/B test models with zero code changes.")

In [ ]:
# =============================================================================
# DEMO: Reusable Classification Function
# =============================================================================

# Pricing per 1K tokens (approximate, on-demand)
PRICING = {
    'anthropic.claude-sonnet-4-5-20250929-v1:0': {'input': 0.00025, 'output': 0.00125, 'name': 'Claude Haiku'},
    'amazon.nova-lite-v1:0': {'input': 0.00006, 'output': 0.00024, 'name': 'Nova Lite'},
    'us.meta.llama3-1-70b-instruct-v1:0': {'input': 0.00099, 'output': 0.00099, 'name': 'Llama 3.1 70B'},
}


def classify_with_bedrock(description, model_id, prompt_template=None):
    """
    Classify a transaction using a Bedrock model.
    Returns: (prediction, latency, input_tokens, output_tokens, cost_usd)
    """
    if prompt_template is None:
        prompt_template = (
            "Classify the following bank transaction as 'fraud' or 'legitimate'. "
            "Respond with only one word: fraud or legitimate.\n\n"
            'Transaction: "{description}"\n\n'
            "Classification:"
        )

    prompt = prompt_template.format(description=description)

    start = time.time()
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{'role': 'user', 'content': [{'text': prompt}]}],
        inferenceConfig={'maxTokens': 20, 'temperature': 0.0}
    )
    elapsed = time.time() - start

    output = response['output']['message']['content'][0]['text'].strip().lower()
    tokens_in = response['usage']['inputTokens']
    tokens_out = response['usage']['outputTokens']

    # Normalize prediction
    if 'fraud' in output:
        prediction = 'fraud'
    elif 'legit' in output:
        prediction = 'legitimate'
    else:
        prediction = output

    # Calculate cost
    pricing = PRICING.get(model_id, {'input': 0, 'output': 0})
    cost = (tokens_in * pricing['input'] + tokens_out * pricing['output']) / 1000

    return prediction, elapsed, tokens_in, tokens_out, cost


# Test it
pred, lat, tin, tout, cost = classify_with_bedrock(
    transaction_descriptions[0]['description'],
    'anthropic.claude-sonnet-4-5-20250929-v1:0'
)
print(f"Prediction: {pred}, Latency: {lat:.2f}s, Cost: ${cost:.6f}")

> **Think About It**: You've now used OpenAI (Week 11), HuggingFace local
> models (Week 12), and Amazon Bedrock (today). At Bread Financial, which
> would you choose for a production fraud detection system? Consider:
> - **Data privacy**: Does transaction data leave your AWS environment?
> - **Vendor lock-in**: What if OpenAI raises prices 10x?
> - **Compliance**: Can you audit which model made which decision?
> - **Cost at scale**: 1 million transactions per month — which is cheapest?

In [ ]:
# =============================================================================
# DEMO: Model Comparison — Quality, Speed, Cost
# =============================================================================
# Run all 3 models on the first 10 transactions to compare.
# (We'll do all 50 in the lab.)

subset = transaction_descriptions[:10]

all_results = []
for model_id, info in PRICING.items():
    model_name = info['name']
    print(f"\nRunning {model_name}...")
    for txn in subset:
        try:
            pred, lat, tin, tout, cost = classify_with_bedrock(txn['description'], model_id)
            all_results.append({
                'id': txn['id'],
                'actual': txn['actual_label'],
                'model': model_name,
                'prediction': pred,
                'latency': lat,
                'tokens_in': tin,
                'tokens_out': tout,
                'cost_usd': cost,
                'correct': pred == txn['actual_label']
            })
            print(f"  ✓ {txn['id']}: {pred}")
        except Exception as e:
            print(f"  ✗ {txn['id']}: {e}")

results_df = pd.DataFrame(all_results)

# Summary per model
print(f"\n{'='*60}")
print(f"{'Model':20s} {'Accuracy':>10s} {'Avg Latency':>12s} {'Avg Cost':>10s}")
print(f"{'-'*60}")
for model_name in results_df['model'].unique():
    mask = results_df['model'] == model_name
    acc = results_df[mask]['correct'].mean()
    avg_lat = results_df[mask]['latency'].mean()
    avg_cost = results_df[mask]['cost_usd'].mean()
    print(f"{model_name:20s} {acc:>9.1%} {avg_lat:>10.2f}s ${avg_cost:>9.6f}")

## Lab 1: Bedrock Model Showdown (15 minutes)

### Your Task

Run all 3 Bedrock models on **all 50 transactions** and create a comprehensive
comparison including accuracy, latency, and cost.

### Steps

1. **Run `classify_with_bedrock`** for each model on all 50 transactions
2. **Calculate per-model metrics**: accuracy, avg latency, total cost, cost per correct prediction
3. **Create a comparison visualization** (grouped bar chart: accuracy + cost)
4. **Compare with Week 12**: How do Bedrock models compare to local models?

### Expected Output

- DataFrame with all results (50 × 3 models = 150 rows)
- Summary table: model, accuracy, avg latency, total cost
- Bar chart comparing accuracy across models
- One-sentence conclusion: which model offers the best value?

### Homework Extension

After class: add the Week 12 local model results to the comparison.
Which is more cost-effective at 10K transactions/month? At 1M?

In [ ]:
# =============================================================================
# SOLUTION: LAB 1 — BEDROCK MODEL SHOWDOWN
# =============================================================================

# Run all models on all 50 transactions
lab1_results = []

for model_id, info in PRICING.items():
    model_name = info['name']
    print(f"\nRunning {model_name} on all 50 transactions...")
    for txn in transaction_descriptions:
        try:
            pred, lat, tin, tout, cost = classify_with_bedrock(txn['description'], model_id)
            lab1_results.append({
                'id': txn['id'],
                'actual': txn['actual_label'],
                'model': model_name,
                'prediction': pred,
                'latency': lat,
                'tokens_in': tin,
                'tokens_out': tout,
                'cost_usd': cost,
                'correct': pred == txn['actual_label']
            })
            print(f"  ✓ {txn['id']}: {pred}")
        except Exception as e:
            print(f"  ✗ {txn['id']}: {e}")
            lab1_results.append({
                'id': txn['id'], 'actual': txn['actual_label'],
                'model': model_name, 'prediction': 'error',
                'latency': 0, 'tokens_in': 0, 'tokens_out': 0,
                'cost_usd': 0, 'correct': False
            })

lab1_df = pd.DataFrame(lab1_results)

# Calculate per-model summary
model_names = lab1_df['model'].unique()
accuracies = []
costs = []
latencies = []
for model_name in model_names:
    mask = lab1_df['model'] == model_name
    accuracies.append(lab1_df[mask]['correct'].mean())
    costs.append(lab1_df[mask]['cost_usd'].sum())
    latencies.append(lab1_df[mask]['latency'].mean())

# Create comparison visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy chart
bars1 = ax1.bar(model_names, accuracies, color=['#3498db', '#e74c3c', '#2ecc71'])
ax1.set_ylabel('Accuracy')
ax1.set_title('Model Accuracy — 50 Fraud Transactions')
ax1.set_ylim(0, 1.1)
for bar, acc in zip(bars1, accuracies):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
             f'{acc:.1%}', ha='center', va='bottom', fontweight='bold')

# Cost chart
bars2 = ax2.bar(model_names, costs, color=['#3498db', '#e74c3c', '#2ecc71'])
ax2.set_ylabel('Total Cost (USD)')
ax2.set_title('Total Cost — 50 Transactions')
for bar, c in zip(bars2, costs):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.0001,
             f'${c:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Print summary
print(f"\n{'='*60}")
print(f"Processed {len(lab1_df)} predictions across {lab1_df['model'].nunique()} models")
for model_name, acc, cost, lat in zip(model_names, accuracies, costs, latencies):
    print(f"  {model_name:20s}: {acc:.1%} accuracy, ${cost:.4f} total, {lat:.2f}s avg latency")
print("\n🎉 Lab 1 complete!")

# Section 2: Bedrock Knowledge Bases — Grounded Answers

So far, our models classify fraud based only on what they learned during
pre-training. But what if we want answers **grounded in our company's actual
fraud detection policies**?

This is where **Knowledge Bases** come in. A Knowledge Base:
1. Stores your documents (policies, procedures, guidelines) in a vector database
2. When you ask a question, it **retrieves relevant chunks** from your documents
3. The LLM generates an answer **citing your actual documents**

![KB vs Raw Model Comparison](charts/kb_comparison.png)

### Our Knowledge Base

Your instructor has pre-configured a Knowledge Base containing:
- **Fraud detection policy documents** (wire transfer rules, card fraud procedures)
- **Transaction pattern guides** (common fraud vs legitimate patterns)
- **Escalation procedures** (what to do when fraud is detected)

In [ ]:
# =============================================================================
# DEMO: Query the Knowledge Base
# =============================================================================
# The Knowledge Base ID was pre-configured by your instructor.
# Students only need to query it — the infra setup was done via Terraform.

KB_ID = os.environ.get("BEDROCK_KB_ID", "QWP1VUKMFS")  # Set by instructor

def query_knowledge_base(question, kb_id=KB_ID, model_id='anthropic.claude-sonnet-4-5-20250929-v1:0'):
    """
    Query the Bedrock Knowledge Base and return the answer with citations.
    """
    model_arn = f'arn:aws:bedrock:{AWS_REGION}::foundation-model/{model_id}'

    response = bedrock_agent_runtime.retrieve_and_generate(
        input={'text': question},
        retrieveAndGenerateConfiguration={
            'type': 'KNOWLEDGE_BASE',
            'knowledgeBaseConfiguration': {
                'knowledgeBaseId': kb_id,
                'modelArn': model_arn
            }
        }
    )

    answer = response['output']['text']
    citations = []
    for citation in response.get('citations', []):
        for ref in citation.get('retrievedReferences', []):
            source = ref.get('location', {}).get('s3Location', {}).get('uri', 'unknown')
            snippet = ref.get('content', {}).get('text', '')[:150]
            citations.append({'source': source, 'snippet': snippet})

    return answer, citations


# Ask about wire transfer policy
question = "What is the policy for international wire transfers over $3,000?"
answer, citations = query_knowledge_base(question)

print(f"Question: {question}")
print(f"\nAnswer: {answer}")
print(f"\nCitations ({len(citations)}):")
for i, c in enumerate(citations, 1):
    print(f"  [{i}] {c['source']}")
    print(f"      \"{c['snippet']}...\"")

In [ ]:
# =============================================================================
# DEMO: Knowledge Base Answer vs Raw Model Answer
# =============================================================================
# Compare: What does Claude say with vs without our documents?

sample = transaction_descriptions[0]  # TXN-001: Wire transfer fraud

# Question about this specific transaction
question = (
    f"Based on our fraud detection policies, analyze this transaction and "
    f"determine if it should be flagged: {sample['description']}"
)

# With Knowledge Base (grounded)
print("=" * 60)
print("WITH KNOWLEDGE BASE (grounded in our documents)")
print("=" * 60)
kb_answer, kb_citations = query_knowledge_base(question)
print(f"Answer: {kb_answer[:300]}")
print(f"Sources cited: {len(kb_citations)}")

# Without Knowledge Base (raw model)
print(f"\n{'=' * 60}")
print("WITHOUT KNOWLEDGE BASE (model's general knowledge only)")
print("=" * 60)
response = bedrock_runtime.converse(
    modelId='anthropic.claude-sonnet-4-5-20250929-v1:0',
    messages=[{'role': 'user', 'content': [{'text': question}]}],
    inferenceConfig={'maxTokens': 300, 'temperature': 0.0}
)
raw_answer = response['output']['message']['content'][0]['text']
print(f"Answer: {raw_answer[:300]}")

print(f"\n💡 The KB answer cites specific policies. The raw answer gives generic advice.")
print(f"   In regulated industries like banking, grounded answers are essential.")

## Lab 2: Query the Knowledge Base (15 minutes)

### Your Task

Write queries to the Knowledge Base for different fraud scenarios and
compare KB-grounded answers against raw model responses.

### Steps

1. **Write 3 different queries** related to fraud detection policies
   (e.g., ATM withdrawal limits, online purchase verification, dispute procedures)
2. **For each query**, get both a KB-grounded answer and a raw model answer
3. **Extract and display citations** for each KB answer
4. **Score each pair**: Is the KB answer more specific? Does it cite policies?

### Expected Output

- 3 query results with KB answer + citations + raw answer
- A simple comparison table: query, KB specificity (1-5), raw specificity (1-5)
- Conclusion: when does the KB add value vs when is the raw model sufficient?

### Homework Extension

After class: try asking the KB questions that it SHOULDN'T be able to answer
(e.g., about a different company's policies). How does it handle out-of-scope
questions? Does it hallucinate or say "I don't know"?

In [ ]:
# =============================================================================
# SOLUTION: LAB 2 — QUERY THE KNOWLEDGE BASE
# =============================================================================

# Write 3 different fraud-related queries
queries = [
    "What are the ATM withdrawal limits and what triggers a fraud alert for ATM transactions?",
    "What verification steps are required for online purchases over $1,000?",
    "What is the escalation procedure when a customer disputes a transaction as fraudulent?",
]

# For each query, get KB answer and raw answer
lab2_results = []
for query in queries:
    # Get KB-grounded answer
    kb_answer, kb_citations = query_knowledge_base(query)

    # Get raw model answer (no KB)
    response = bedrock_runtime.converse(
        modelId='anthropic.claude-sonnet-4-5-20250929-v1:0',
        messages=[{'role': 'user', 'content': [{'text': query}]}],
        inferenceConfig={'maxTokens': 300, 'temperature': 0.0}
    )
    raw_answer = response['output']['message']['content'][0]['text']

    lab2_results.append({
        'query': query,
        'kb_answer': kb_answer,
        'citations': kb_citations,
        'raw_answer': raw_answer,
    })

# Display results
for i, r in enumerate(lab2_results, 1):
    print(f"{'='*60}")
    print(f"Query {i}: {r['query']}")
    print(f"\n📚 KB Answer (grounded):")
    print(f"  {r['kb_answer'][:200]}...")
    print(f"  Citations: {len(r['citations'])}")
    for c in r['citations']:
        print(f"    - {c['source'].split('/')[-1]}")
    print(f"\n🤖 Raw Answer (no KB):")
    print(f"  {r['raw_answer'][:200]}...")
    print()

# Comparison table
print(f"\n{'='*60}")
print("Comparison Summary")
print(f"{'Query':<40s} {'KB Citations':>12s}")
for r in lab2_results:
    print(f"  {r['query'][:38]:<40s} {len(r['citations']):>8d}")

print("\n🎉 Lab 2 complete!")

> **Think About It**: Knowledge Bases give answers grounded in your actual
> documents. But they add latency (retrieval + generation) and cost (vector
> storage + embedding). For fraud detection at Bread Financial: would you use
> a KB for real-time transaction scoring, or only for analyst-facing tools
> where accuracy matters more than speed?

# Section 3: Synthetic Data Generation

Now that we know which Bedrock model performs best (Section 1) and how to
ground answers in policy documents (Section 2), we'll tackle the next piece
of the pipeline: **creating training data for Week 14's fine-tuning**.

One of the most powerful LLM applications for data scientists: **generating
synthetic training data**. This solves several real problems:

| Problem | How Synthetic Data Helps |
|---------|------------------------|
| **Small dataset** | Generate 100x more labeled examples |
| **Class imbalance** | Generate more minority-class examples |
| **Privacy concerns** | Synthetic data doesn't contain real customer info |
| **New categories** | Generate examples for fraud types you haven't seen yet |

![Synthetic Data Pipeline](charts/synthetic_pipeline.png)

We'll use Bedrock to generate synthetic fraud and legitimate transaction
descriptions, validate their quality, and save a CSV that **Week 14 will
load directly** to fine-tune a fraud classifier.

In [ ]:
# =============================================================================
# DEMO: Generate Synthetic Fraud Transactions
# =============================================================================

# Note: We frame the prompt as educational/training data to avoid content filters.
# Claude may refuse "generate fraud descriptions" without this framing.
SYNTH_FRAUD_PROMPT = """You are helping a data science team build a fraud detection training dataset
for an educational workshop. Generate 5 FICTIONAL bank fraud transaction descriptions
that a fraud analyst might see in their case management system.

Each should describe a DIFFERENT type of fraud (e.g., wire fraud, card theft,
account takeover, identity theft, card testing).

For each transaction, provide:
- A 2-3 sentence description of what happened
- Include specific details: dollar amounts, times, locations, patterns

Format your response as a JSON array:
[
  {"description": "...", "fraud_type": "wire_fraud"},
  {"description": "...", "fraud_type": "card_theft"},
  ...
]

Make each description realistic and detailed, as if from a bank's internal system."""

response = bedrock_runtime.converse(
    modelId='anthropic.claude-sonnet-4-5-20250929-v1:0',
    messages=[{'role': 'user', 'content': [{'text': SYNTH_FRAUD_PROMPT}]}],
    inferenceConfig={'maxTokens': 1000, 'temperature': 0.7}
)

raw_output = response['output']['message']['content'][0]['text']
print("Raw LLM output:")
print(raw_output[:500])

# Parse JSON from response
try:
    # Strip markdown code fences (Claude often wraps JSON in ```json...```)
    cleaned = re.sub(r'```json\s*', '', raw_output)
    cleaned = re.sub(r'```\s*', '', cleaned)
    # Find JSON array in the response
    json_match = re.search(r'\[.*\]', cleaned, re.DOTALL)
    if json_match:
        synthetic_fraud = json.loads(json_match.group())
        print(f"\n✅ Generated {len(synthetic_fraud)} synthetic fraud transactions:")
        for i, txn in enumerate(synthetic_fraud, 1):
            print(f"\n  [{i}] Type: {txn.get('fraud_type', 'unknown')}")
            print(f"      {txn['description'][:100]}...")
    else:
        print("\n❌ Could not find JSON in response")
except json.JSONDecodeError as e:
    print(f"\n❌ JSON parsing error: {e}")

In [ ]:
# =============================================================================
# DEMO: Generate Synthetic Legitimate Transactions
# =============================================================================

SYNTH_LEGIT_PROMPT = """Generate 5 realistic LEGITIMATE bank transaction descriptions.
Each should describe a DIFFERENT type of normal transaction (e.g., subscription payment,
grocery shopping, payroll deposit, utility bill, online purchase).

For each transaction, provide:
- A 2-3 sentence description with specific details
- Include: amounts, merchants, frequency patterns, payment methods

Format as JSON array:
[
  {"description": "...", "transaction_type": "subscription"},
  {"description": "...", "transaction_type": "grocery"},
  ...
]

Make them sound like normal, everyday banking activity."""

response = bedrock_runtime.converse(
    modelId='anthropic.claude-sonnet-4-5-20250929-v1:0',
    messages=[{'role': 'user', 'content': [{'text': SYNTH_LEGIT_PROMPT}]}],
    inferenceConfig={'maxTokens': 1000, 'temperature': 0.7}
)

raw_output = response['output']['message']['content'][0]['text']

try:
    # Strip markdown code fences (Claude often wraps JSON in ```json...```)
    cleaned = re.sub(r'```json\s*', '', raw_output)
    cleaned = re.sub(r'```\s*', '', cleaned)
    json_match = re.search(r'\[.*\]', cleaned, re.DOTALL)
    if json_match:
        synthetic_legit = json.loads(json_match.group())
        print(f"✅ Generated {len(synthetic_legit)} synthetic legitimate transactions:")
        for i, txn in enumerate(synthetic_legit, 1):
            print(f"\n  [{i}] Type: {txn.get('transaction_type', 'unknown')}")
            print(f"      {txn['description'][:100]}...")
except json.JSONDecodeError as e:
    print(f"❌ JSON parsing error: {e}")

In [ ]:
# =============================================================================
# DEMO: Quality Validation of Synthetic Data
# =============================================================================
# Before using synthetic data for training, we need to check:
# 1. Are descriptions diverse (not repetitive)?
# 2. Are they realistic (contain specific details)?
# 3. Can our classifier distinguish them from real data?

def validate_synthetic_batch(transactions, label):
    """Run basic quality checks on a batch of synthetic transactions."""
    descriptions = [t['description'] for t in transactions]

    # Check 1: Diversity — no two descriptions should be too similar
    from difflib import SequenceMatcher
    similarities = []
    for i in range(len(descriptions)):
        for j in range(i+1, len(descriptions)):
            sim = SequenceMatcher(None, descriptions[i], descriptions[j]).ratio()
            similarities.append(sim)

    avg_similarity = np.mean(similarities) if similarities else 0
    max_similarity = max(similarities) if similarities else 0

    # Check 2: Detail — descriptions should have numbers, locations, times
    has_amount = sum(1 for d in descriptions if re.search(r'\$[\d,]+', d))
    has_time = sum(1 for d in descriptions if re.search(r'\d{1,2}:\d{2}|AM|PM|morning|evening|night', d, re.IGNORECASE))
    has_location = sum(1 for d in descriptions if re.search(r'at |in |from |store|bank|ATM', d, re.IGNORECASE))

    # Check 3: Length — should be descriptive, not too short
    avg_length = np.mean([len(d.split()) for d in descriptions])

    print(f"Quality Report for {label} transactions ({len(transactions)} items):")
    print(f"  Diversity:  avg similarity = {avg_similarity:.2f} (lower is better, <0.3 ideal)")
    print(f"  Max pair similarity: {max_similarity:.2f} (should be <0.5)")
    print(f"  Has $ amounts: {has_amount}/{len(descriptions)}")
    print(f"  Has time refs:  {has_time}/{len(descriptions)}")
    print(f"  Has locations:  {has_location}/{len(descriptions)}")
    print(f"  Avg word count: {avg_length:.0f} words")

    return avg_similarity < 0.3 and avg_length > 15


# Validate our demo batches
print("=" * 50)
fraud_ok = validate_synthetic_batch(synthetic_fraud, "FRAUD")
print()
legit_ok = validate_synthetic_batch(synthetic_legit, "LEGITIMATE")

if fraud_ok and legit_ok:
    print("\n✅ Quality checks passed!")
else:
    print("\n⚠️ Some quality checks failed — consider regenerating with different prompts")

## Lab 3: Generate Fraud Training Data (15 minutes)

### Your Task

Generate a synthetic training dataset of **50 transactions** (25 fraud, 25
legitimate) that we'll use in **Week 14** to fine-tune a fraud classifier.

### Steps

1. **Generate 25 fraud transactions** in 5 batches of 5, each batch covering
   different fraud types (wire fraud, card theft, account takeover, identity
   theft, card testing, etc.)
2. **Generate 25 legitimate transactions** in 5 batches of 5, each covering
   different transaction types (subscriptions, groceries, payroll, utilities, etc.)
3. **Run quality validation** on the full dataset
4. **Combine into a DataFrame** with columns: description, label, fraud_type/transaction_type
5. **Save to CSV** for use in Week 14

### Expected Output

- DataFrame with 50 rows (25 fraud, 25 legitimate)
- Quality validation passing for both classes
- CSV saved as `synthetic_fraud_data.csv`

### Homework Extension

After class: generate 200 more examples. Experiment with temperature (0.5 vs 0.9)
and see how it affects diversity. Try generating "edge case" transactions that
are ambiguous — would these be useful for training?

In [ ]:
# =============================================================================
# SOLUTION: LAB 3 — GENERATE FRAUD TRAINING DATA
# =============================================================================

FRAUD_TYPES = ['wire_fraud', 'card_theft', 'account_takeover', 'identity_theft', 'card_testing']
LEGIT_TYPES = ['subscription', 'grocery', 'payroll', 'utility', 'online_purchase']

all_synthetic = []

# Generate 5 batches of 5 fraud transactions
for fraud_type in FRAUD_TYPES:
    # Educational framing avoids content filters
    prompt = f"""You are helping a data science team build a fraud detection training dataset
for an educational workshop. Generate 5 FICTIONAL bank fraud transaction descriptions of type: {fraud_type}.
Each should be 2-3 sentences with specific details (amounts, times, locations).
Format as JSON array: [{{"description": "...", "fraud_type": "{fraud_type}"}}]"""

    response = bedrock_runtime.converse(
        modelId='anthropic.claude-sonnet-4-5-20250929-v1:0',
        messages=[{'role': 'user', 'content': [{'text': prompt}]}],
        inferenceConfig={'maxTokens': 1000, 'temperature': 0.7}
    )
    raw = response['output']['message']['content'][0]['text']
    try:
        # Strip markdown code fences before parsing
        cleaned = re.sub(r'```json\s*', '', raw)
        cleaned = re.sub(r'```\s*', '', cleaned)
        match = re.search(r'\[.*\]', cleaned, re.DOTALL)
        if match:
            batch = json.loads(match.group())
            for item in batch:
                item['label'] = 'fraud'
            all_synthetic.extend(batch)
            print(f"  ✓ Generated {len(batch)} {fraud_type} transactions")
    except Exception as e:
        print(f"  ✗ Error generating {fraud_type}: {e}")

# Generate 5 batches of 5 legitimate transactions
for legit_type in LEGIT_TYPES:
    prompt = f"""Generate 5 realistic LEGITIMATE bank transaction descriptions of type: {legit_type}.
Each should be 2-3 sentences with specific details (amounts, merchants, patterns).
Format as JSON array: [{{"description": "...", "transaction_type": "{legit_type}"}}]"""

    response = bedrock_runtime.converse(
        modelId='anthropic.claude-sonnet-4-5-20250929-v1:0',
        messages=[{'role': 'user', 'content': [{'text': prompt}]}],
        inferenceConfig={'maxTokens': 1000, 'temperature': 0.7}
    )
    raw = response['output']['message']['content'][0]['text']
    try:
        # Strip markdown code fences before parsing
        cleaned = re.sub(r'```json\s*', '', raw)
        cleaned = re.sub(r'```\s*', '', cleaned)
        match = re.search(r'\[.*\]', cleaned, re.DOTALL)
        if match:
            batch = json.loads(match.group())
            for item in batch:
                item['label'] = 'legitimate'
            all_synthetic.extend(batch)
            print(f"  ✓ Generated {len(batch)} {legit_type} transactions")
    except Exception as e:
        print(f"  ✗ Error generating {legit_type}: {e}")

# Combine into DataFrame
synthetic_df = pd.DataFrame(all_synthetic)
synthetic_df = synthetic_df[['description', 'label']].copy()
synthetic_df['id'] = [f'SYN-{i+1:03d}' for i in range(len(synthetic_df))]

# Quality validation
fraud_items = [{'description': d} for d in synthetic_df[synthetic_df['label'] == 'fraud']['description']]
legit_items = [{'description': d} for d in synthetic_df[synthetic_df['label'] == 'legitimate']['description']]
print()
validate_synthetic_batch(fraud_items, "FRAUD")
print()
validate_synthetic_batch(legit_items, "LEGITIMATE")

# Save to CSV for Week 14
synthetic_df.to_csv('synthetic_fraud_data.csv', index=False)
print(f"\n✅ Generated {len(synthetic_df)} synthetic transactions")
print(f"   Fraud: {(synthetic_df['label'] == 'fraud').sum()}")
print(f"   Legitimate: {(synthetic_df['label'] == 'legitimate').sum()}")
print(f"   Saved to: synthetic_fraud_data.csv")
print(f"\n🎉 Lab 3 complete! Data saved for Week 14.")

# Section 4: LLM-Assisted EDA + DeepEval

We've generated synthetic data — but how do we know it's good enough to train
on? This final section closes the loop: use LLMs to explore the data, and
**DeepEval** to systematically evaluate output quality.

Two powerful techniques:

1. **LLM-Assisted EDA**: Ask an LLM to analyze your data and suggest insights
2. **DeepEval**: A Python library for systematically evaluating LLM outputs
   (like Week 12's manual evaluation framework, but production-grade)

> **Note on Structured Outputs**: Newer Bedrock models (Claude 4.5+) support
> `outputConfig.textFormat` to enforce JSON schema responses — similar to
> OpenAI's `response_format`. Claude 3 Haiku doesn't support this yet, so
> we parse JSON from free-text responses using regex. In production with
> newer models, you'd use structured outputs for guaranteed valid JSON.

In [ ]:
# =============================================================================
# DEMO: LLM-Assisted Exploratory Data Analysis
# =============================================================================
# Feed a dataset summary to Bedrock and ask for analysis suggestions.

# Create a summary of our fraud dataset
dataset_summary = f"""Dataset: Bank Fraud Transactions
Total transactions: {len(df)}
Fraud: {(df['actual_label'] == 'fraud').sum()}
Legitimate: {(df['actual_label'] == 'legitimate').sum()}

Sample fraud transaction:
  "{df[df['actual_label'] == 'fraud'].iloc[0]['description']}"

Sample legitimate transaction:
  "{df[df['actual_label'] == 'legitimate'].iloc[0]['description']}"

Columns: id, description (text), actual_label (fraud/legitimate)
"""

eda_prompt = f"""You are a data scientist analyzing a fraud detection dataset.
Here is a summary of the data:

{dataset_summary}

Suggest 3 specific analyses I could run on this text data to better understand
fraud patterns. For each analysis, provide the pandas/Python code to execute it.
Focus on text features that distinguish fraud from legitimate transactions."""

response = bedrock_runtime.converse(
    modelId='anthropic.claude-sonnet-4-5-20250929-v1:0',
    messages=[{'role': 'user', 'content': [{'text': eda_prompt}]}],
    inferenceConfig={'maxTokens': 1000, 'temperature': 0.3}
)

eda_suggestions = response['output']['message']['content'][0]['text']
print("LLM EDA Suggestions:")
print("=" * 50)
print(eda_suggestions)

In [ ]:
# =============================================================================
# DEMO: DeepEval — Production-Grade LLM Evaluation
# =============================================================================
# In Week 12 we built evaluation by hand (extract_label, check_format, etc.).
# DeepEval does this systematically with LLM-as-a-judge.

from deepeval.models import AmazonBedrockModel
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# Use Bedrock Claude as the judge (no OpenAI key needed!)
# On SageMaker, it picks up credentials from the execution role automatically
judge_model = AmazonBedrockModel(
    model_id='anthropic.claude-sonnet-4-5-20250929-v1:0',
    region=AWS_REGION,
)

# Create a correctness metric using G-Eval
correctness_metric = GEval(
    name="Fraud Classification Correctness",
    criteria="Determine if the 'actual output' correctly classifies the transaction as fraud or legitimate based on the 'expected output'.",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    model=judge_model,
)

# Test on one transaction
sample = transaction_descriptions[0]
pred, _, _, _, _ = classify_with_bedrock(sample['description'], 'anthropic.claude-sonnet-4-5-20250929-v1:0')

test_case = LLMTestCase(
    input=sample['description'],
    actual_output=pred,
    expected_output=sample['actual_label']
)

correctness_metric.measure(test_case)
print(f"Transaction: {sample['id']}")
print(f"Prediction:  {pred}")
print(f"Actual:      {sample['actual_label']}")
print(f"DeepEval Score:  {correctness_metric.score:.2f}")
print(f"Reason: {correctness_metric.reason}")
print(f"\n💡 DeepEval uses an LLM as a judge to evaluate other LLM outputs.")
print(f"   This scales much better than our manual evaluation from Week 12.")

> **Think About It**: DeepEval uses an LLM to judge another LLM's output.
> Is this circular? What if the judge model has the same biases as the model
> being evaluated? In what cases would you still need human evaluation?

# Summary: What We Learned Today

## Key Takeaways

### Bedrock Converse API
- **Unified API** across Claude, Llama, Nova, Titan — just change `modelId`
- Returns structured metadata: token counts, cost tracking, stop reasons
- Enterprise-grade: data stays in AWS, audit trail, compliance

### Knowledge Bases
- Grounded answers backed by your actual documents
- Retrieval + generation in one API call
- Citations tell you exactly which document the answer came from

### Synthetic Data Generation
- LLMs can generate realistic training data at scale
- Key checks: diversity, specificity, realism
- Week 14 will use this data for fine-tuning

### LLM-Assisted EDA + DeepEval
- LLMs can suggest analyses and generate code
- DeepEval provides production-grade evaluation with LLM-as-a-judge
- Bedrock models can serve as both the evaluated model and the judge

## The Bigger Picture

![Week Progression](charts/week_progression.png)

| Week | What We Did |
|------|------------|
| 11 | Cloud APIs (OpenAI, Anthropic), prompting patterns |
| 12 | Local HuggingFace models, evaluation, error analysis |
| **13** | **Bedrock managed models, Knowledge Bases, synthetic data, DeepEval** |
| 14 | Fine-tune a fraud classifier using our synthetic data |

# Homework & Optional Labs

## Homework (Complete before next session)

### Homework 1: Cost Calculator
Calculate the monthly cost of processing 10K, 100K, and 1M transactions
with each Bedrock model. Create a table showing cost breakdowns.
At what volume does the cheapest model become the best choice?

### Homework 2: Knowledge Base Edge Cases
Query the KB with 5 questions it SHOULDN'T be able to answer.
How does it handle out-of-scope queries? Does it hallucinate or refuse?

### Homework 3: Advanced Synthetic Data
Generate 200 more synthetic transactions (100 fraud, 100 legitimate).
Experiment with temperature=0.5 vs 0.9. Which produces more diverse data?
Save the combined dataset for Week 14.

## Optional Lab

### Optional: Advanced DeepEval Metrics
See `week_13_optional_deepeval_advanced.ipynb` for:
- Answer relevancy metrics for KB responses
- Faithfulness metrics (does the answer match the sources?)
- Custom evaluation criteria with G-Eval
- Building an automated evaluation pipeline

# Great Work Today!

You've completed Week 13 of the AI for Data Scientists Academy.

**What you accomplished:**
- Called multiple Bedrock models with the unified Converse API
- Queried a Knowledge Base for grounded, document-backed answers
- Generated synthetic training data for downstream ML tasks
- Evaluated LLM outputs with DeepEval

## Coming Up Next

- **Week 14**: Fine-tune a DistilBERT fraud classifier using today's synthetic data
- **Weeks 15-16**: Agentic AI — ReAct agents, LangChain, multi-agent systems
- **Weeks 17-18**: RAG — build your own retrieval pipeline (beyond Knowledge Bases)

## Resources

- [Amazon Bedrock Documentation](https://docs.aws.amazon.com/bedrock/)
- [Bedrock Converse API Reference](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_Converse.html)
- [DeepEval Documentation](https://deepeval.com/docs/getting-started)
- [DeepEval + Bedrock Integration](https://deepeval.com/integrations/models/amazon-bedrock)

See you in Week 14!